#### In Class Assignment 2026.04.28

In [1]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, classification_report
)

In [2]:
# load irrigation data, inspect it, and split it
df = pd.read_csv("test.csv")

print(df.head())
print()
print(df["Irrigation_Type"].value_counts())
print()
print(df.isna().sum())

X = df.drop(["id", "Irrigation_Type"], axis=1)
y = df["Irrigation_Type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

       id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0  630000      Silt     6.36          26.19            0.59   
1  630001      Clay     5.87           9.88            1.18   
2  630002     Sandy     6.22          26.55            0.96   
3  630003      Clay     7.68          53.58            0.83   
4  630004     Loamy     5.23          59.02            0.54   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     2.81          17.83     30.24      1533.38   
1                     3.26          21.18     78.07       576.05   
2                     0.85          26.87     60.35       545.30   
3                     0.55          41.74     36.05      1211.03   
4                     2.11          41.08     52.47      1321.91   

   Sunlight_Hours  Wind_Speed_kmh Crop_Type Crop_Growth_Stage  Season  \
0            5.40            3.00     Maize            Sowing    Rabi   
1            7.22           15.88    Cotton            Sowing    R

In [3]:
# identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:", numeric_features)
print("categorical columns:", categorical_features)

numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Water_Source', 'Mulching_Used', 'Region']


### Defining helper functions

In [4]:
# helper for converting sparse matrices to dense arrays
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

### Building the preprocessing pipeline and Gaussian Naive Bayes model

In [5]:
# define preprocessors
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# build gaussian naive bayes pipeline
gnb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", FunctionTransformer(to_dense)),
    ("model", GaussianNB())
])

### Cross validation

In [6]:
# cross validate the naive bayes model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_validate(
    gnb_pipeline, X_train, y_train, cv=cv,
    scoring=["accuracy", "balanced_accuracy"]
)

print("cv accuracy:         ", round(cv_scores["test_accuracy"].mean(), 4))
print("cv balanced accuracy:", round(cv_scores["test_balanced_accuracy"].mean(), 4))

cv accuracy:          0.2831
cv balanced accuracy: 0.281


### Fitting model and generating predicted probabilities

In [7]:
# fit model on training data and generate predicted probabilities on test set
gnb_pipeline.fit(X_train, y_train)

probs = gnb_pipeline.predict_proba(X_test)
classes = gnb_pipeline.classes_

print("classes:", classes)
print("probability array shape:", probs.shape)
print()
print("sample predicted probabilities (first 5 rows):")
print(pd.DataFrame(probs, columns=classes).head())

classes: ['Canal' 'Drip' 'Rainfed' 'Sprinkler']
probability array shape: (54000, 4)

sample predicted probabilities (first 5 rows):
      Canal      Drip   Rainfed  Sprinkler
0  0.248894  0.146688  0.283068   0.321350
1  0.302048  0.194302  0.234315   0.269334
2  0.307312  0.218981  0.250393   0.223315
3  0.413945  0.172124  0.225165   0.188765
4  0.294340  0.283500  0.211318   0.210842


### Baseline predictions using default rule (highest probability class)

In [8]:
# baseline predictions: assign the class with the highest predicted probability
baseline_preds = gnb_pipeline.predict(X_test)

print("baseline classification report:")
print(classification_report(y_test, baseline_preds, digits=4))

print("baseline accuracy:         ", round(accuracy_score(y_test, baseline_preds), 4))
print("baseline balanced accuracy:", round(balanced_accuracy_score(y_test, baseline_preds), 4))

baseline classification report:
              precision    recall  f1-score   support

       Canal     0.3000    0.3838    0.3368     13932
        Drip     0.2645    0.1897    0.2210     12875
     Rainfed     0.2748    0.2534    0.2637     13279
   Sprinkler     0.2870    0.3031    0.2949     13914

    accuracy                         0.2847     54000
   macro avg     0.2816    0.2825    0.2791     54000
weighted avg     0.2820    0.2847    0.2804     54000

baseline accuracy:          0.2847
baseline balanced accuracy: 0.2825


### Threshold-based predictions for Rainfed class

In [9]:
# focus class: Rainfed
# get the index of Rainfed in the classes array
rainfed_idx = list(classes).index("Rainfed")
rainfed_probs = probs[:, rainfed_idx]

# scan thresholds to find a good f1 for Rainfed
thresholds = np.linspace(0.10, 0.90, 17)
rows = []
for t in thresholds:
    # assign Rainfed if its probability exceeds threshold, otherwise use argmax over remaining classes
    preds = []
    for i in range(len(probs)):
        if probs[i, rainfed_idx] >= t:
            preds.append("Rainfed")
        else:
            # pick the highest probability among non-Rainfed classes
            masked = probs[i].copy()
            masked[rainfed_idx] = -1
            preds.append(classes[np.argmax(masked)])
    preds = np.array(preds)
    rows.append({
        "threshold": round(t, 2),
        "rainfed_f1": round(f1_score(y_test, preds, labels=["Rainfed"], average="macro", zero_division=0), 4),
        "rainfed_recall": round(recall_score(y_test, preds, labels=["Rainfed"], average="macro", zero_division=0), 4),
        "rainfed_precision": round(precision_score(y_test, preds, labels=["Rainfed"], average="macro", zero_division=0), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_test, preds), 4)
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df.sort_values("rainfed_f1", ascending=False).head(10))

   threshold  rainfed_f1  rainfed_recall  rainfed_precision  balanced_accuracy
0       0.10      0.3947          1.0000             0.2459             0.2500
1       0.15      0.3946          0.9829             0.2468             0.2520
2       0.20      0.3868          0.8179             0.2533             0.2656
3       0.25      0.3459          0.4971             0.2652             0.2794
4       0.30      0.2264          0.1898             0.2807             0.2817
5       0.35      0.0765          0.0439             0.2970             0.2794
6       0.40      0.0123          0.0063             0.3180             0.2777
7       0.45      0.0006          0.0003             0.4444             0.2774
8       0.50      0.0000          0.0000             0.0000             0.2774
9       0.55      0.0000          0.0000             0.0000             0.2774


In [10]:
# apply chosen threshold of 0.20 for Rainfed class
chosen_threshold = 0.20

threshold_preds = []
for i in range(len(probs)):
    if probs[i, rainfed_idx] >= chosen_threshold:
        threshold_preds.append("Rainfed")
    else:
        masked = probs[i].copy()
        masked[rainfed_idx] = -1
        threshold_preds.append(classes[np.argmax(masked)])
threshold_preds = np.array(threshold_preds)

print(f"threshold predictions (threshold={chosen_threshold}) classification report:")
print(classification_report(y_test, threshold_preds, digits=4))

print("threshold balanced accuracy:", round(balanced_accuracy_score(y_test, threshold_preds), 4))

threshold predictions (threshold=0.2) classification report:
              precision    recall  f1-score   support

       Canal     0.3179    0.1218    0.1761     13932
        Drip     0.2666    0.0713    0.1125     12875
     Rainfed     0.2533    0.8179    0.3868     13279
   Sprinkler     0.3069    0.0515    0.0882     13914

    accuracy                         0.2628     54000
   macro avg     0.2862    0.2656    0.1909     54000
weighted avg     0.2870    0.2628    0.1901     54000

threshold balanced accuracy: 0.2656


### Discussion

**Class selected:** I focused on Rainfed. All four classes are roughly balanced (~24-25% each), but Rainfed fields get no active irrigation, so a false negative means wasting water on a field that doesn't need it. I used F1 as my metric since it balances precision and recall into one number.

**Threshold chosen:** I chose 0.20. In a 4-class problem, the default argmax only needs to beat ~25% to win, so lowering the Rainfed threshold to 0.20 means we flag it whenever the model gives it at least 20% probability, which catches more Rainfed cases before another class edges it out.

**How the metric changed:** Baseline Rainfed F1 is 0.2637 (precision 0.27, recall 0.25). At threshold 0.20, F1 goes up to 0.3868, mostly because recall jumps from 0.25 to 0.82. The model catches a lot more true Rainfed fields. Precision barely moves (0.27 to 0.25) since we're pulling in a few extra false positives.

**Tradeoff observed:** Lowering the Rainfed threshold takes predictions away from the other three classes. Canal, Drip, and Sprinkler all see their recall drop significantly since many of those examples now get labeled Rainfed. Overall balanced accuracy actually falls from 0.2825 to 0.2656 because the Rainfed gains don't make up for the losses elsewhere. Basically, boosting one class's recall in a multiclass setting hurts the rest.

**Naive Bayes vs. other models:** GNB is performing near-randomly here (~28% accuracy vs. a ~25% random baseline), which isn't surprising. GNB assumes all features are independent and Gaussian, and neither is true for this data. The continuous features probably aren't Gaussian within each class, and the OHE categorical features are binary indicators that definitely aren't. My other models (XGBoost, Random Forest) handle mixed feature types and interactions much better and would be expected to do significantly better here. GNB is a useful quick baseline but not a real contender for this dataset.